In [1]:
cfg= {
    "vocab_size": 1000,
    "hidden_size": 256,
    
    "num_hidden_layers": 6,
    "num_attention_heads": 8,
    "num_key_value_heads": 8,

    "intermediate_size": 1024,

    "max_position_embeddings": 128,

    "eps": 1e-6,
    "rope_theta": 10000.0,

    "dropout": 0.0,
    "bias": False,
}

In [2]:
import torch
from BPE import BPE
from data_loader import load_dataset_,create_loader

ds=load_dataset_()
corpus = "\n".join(sample["text"] for sample in ds)
print(len(corpus))
tokenizer=BPE(vocab_size=1000)
tokenizer.train(corpus)
dataloader=create_loader(corpus,tokenizer,max_length=128,stride=128,shuffle=False,batch_size=8)

data_iter=iter(dataloader)
ip,tgt=next(data_iter)
print("\nInputs\n",ip)
print("\ntarget\n",tgt)

print("\nInput shape:\n",ip.shape)
print("\nTarget shape:\n",tgt.shape)

token_ids=tokenizer.encode(corpus)
torch.save(token_ids,"token_ids.pt")

5905283

Inputs
 tensor([[701, 687, 620,  ..., 428, 497, 417],
        [847, 595, 834,  ..., 538, 801, 482],
        [782, 593,  22,  ..., 482, 531, 748],
        ...,
        [333, 573, 469,  ..., 542, 529, 855],
        [828,   1, 608,  ..., 773, 862, 417],
        [707, 333, 351,  ..., 608, 487,  36]])

target
 tensor([[687, 620, 608,  ..., 497, 417, 847],
        [595, 834, 743,  ..., 801, 482, 782],
        [593,  22, 588,  ..., 531, 748, 872],
        ...,
        [573, 469, 525,  ..., 529, 855, 828],
        [  1, 608, 487,  ..., 862, 417, 707],
        [333, 351, 426,  ..., 487,  36, 619]])

Input shape:
 torch.Size([8, 128])

Target shape:
 torch.Size([8, 128])


In [3]:
from Embedding import Embedding
embedding_layer=Embedding(cfg)
emb=embedding_layer(ip)
from RMSNorm import RMSNorm
rms=RMSNorm(cfg)
print(emb.shape)
print(rms(emb).shape)

torch.Size([8, 128, 256])
torch.Size([8, 128, 256])


In [4]:
from ROPE import ROPE
rope = ROPE(cfg)

x = torch.randn(
    2,
    128,
    cfg["num_attention_heads"],
    cfg["hidden_size"] // cfg["num_attention_heads"]
)

out = rope(x)

print(x.shape)
print(out.shape)

torch.Size([2, 128, 8, 32])
torch.Size([2, 128, 8, 32])


In [5]:
import torch
import torch.nn as nn
from MHA import MHA


In [6]:
norma=rms(emb)
mha=MHA(cfg)
mha(norma).shape

torch.Size([8, 128, 256])

In [7]:
from SwiGLU import SwiGLU
ffn = SwiGLU(cfg)

x = torch.randn(8, 128, cfg["hidden_size"])

out = ffn(x)

print(out.shape)

torch.Size([8, 128, 256])


In [8]:
from llama import Llama

model = Llama(cfg)
model(ip).shape

torch.Size([8, 128, 1000])

In [9]:
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} parameters")

6,564,608 parameters


In [13]:
logits=model(ip)
logits.shape

torch.Size([8, 128, 1000])

In [21]:
logits=logits.view(-1,cfg["vocab_size"])
tgt=tgt.view(-1)
logits.shape,tgt.shape

torch.Size([8, 128])


(torch.Size([1024, 1000]), torch.Size([1024]))

In [17]:
loss_fn=nn.CrossEntropyLoss()
loss=loss_fn(logits,tgt)
loss

tensor(6.9463, grad_fn=<NllLossBackward0>)

In [30]:
model=Llama(cfg)
optim=torch.optim.AdamW(model.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()
model.train();

In [25]:
from torch.utils.data import random_split,DataLoader
dataset = dataloader.dataset

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

In [48]:
from tqdm import tqdm
import os
epochs=2
best_val_loss=float("inf")
start=0
if os.path.exists("latest_model.pt"):

    checkpoint = torch.load("latest_model.pt")

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optim.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    start = checkpoint["epoch"] + 1

    best_val_loss = checkpoint["val_loss"]

    print(f"Resuming from epoch {start}")
device="cuda" if torch.cuda.is_available() else "cpu"
mode=model.to(device)
for epoch in range(start,epochs):
    epoch_loss_train=0
    epoch_loss_val=0
    model.train()
    for ip,tgt in tqdm(train_loader):
        ip=ip.to(device)
        tgt=tgt.to(device)
        logits=model(ip)
        logits=logits.view(-1,cfg["vocab_size"])
        tgt=tgt.view(-1)
        loss=loss_fn(logits,tgt)
        optim.zero_grad()
        loss.backward()
        optim.step()
        epoch_loss_train+=loss.item()
    model.eval()
    with torch.inference_mode():
        for ip,tgt in tqdm(val_loader):
            ip=ip.to(device)
            tgt=tgt.to(device)
            logits=model(ip)
            logits=logits.view(-1,cfg["vocab_size"])
            tgt=tgt.view(-1)
            loss=loss_fn(logits,tgt)
            epoch_loss_val+=loss.item()
    epoch_loss_train/=len(train_loader)
    epoch_loss_val/=len(val_loader)
    torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optim.state_dict(),
            "train_loss": epoch_loss_train,
            "val_loss": epoch_loss_val,
        }, "latest_model.pt")
    if epoch_loss_val<epoch_loss_val:
        best_vl_loss=epoch_loss_val
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optim.state_dict(),
            "train_loss": epoch_loss_train,
            "val_loss": epoch_loss_val,
        }, "best_model.pt")
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss_train:.4f} | Val Loss: {epoch_loss_val:.4f}")

100%|██████████| 215/215 [00:21<00:00,  9.80it/s]


Epoch 1/2 | Train Loss: 2.5656 | Val Loss: 2.4996


100%|██████████| 215/215 [00:21<00:00,  9.85it/s]

Epoch 2/2 | Train Loss: 2.3681 | Val Loss: 2.3932


In [49]:
def generate_text_greedy(prompt,max_new_tokens,tokenizer,model):
    model.eval()
    ip=torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits=model(ip)
        id_=torch.argmax(logits[:,-1,:],dim=-1)
        ip=torch.cat((ip,id_.unsqueeze(1)),dim=1)
    return tokenizer.decode(ip.squeeze(0).tolist())

In [50]:
print(generate_text_greedy(
    prompt="ఒక",
    max_new_tokens=20,
    tokenizer=tokenizer,
    model=model
))

ఒక చిన్న ఎలుక ఒక రోజు బజారులో ఆడుకుంటుండగా, ఒక పెద్ద బంతి దొరికింది.


In [51]:
def generate_text(prompt,max_new_tokens,tokenizer,model,temp):
    model.eval()
    ip=torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits=model(ip)
        logits=logits[:, -1, :]
        logits/=temp
        logits=torch.softmax(logits,dim=-1)
        id_=torch.multinomial(logits,num_samples=1)
        ip=torch.cat((ip,id_),dim=1)
    return tokenizer.decode(ip.squeeze(0).tolist())

In [57]:
for _ in range(5):
    print(generate_text(
        prompt="ఒక",
        max_new_tokens=30,
        tokenizer=tokenizer,
        model=model,
        temp=0.8
    ))
    print("-" * 40)

ఒక చిన్న బొట్టు కింద ఆడుకుంటుండగా, రెండు అందమైన చెవిపోగులు కనిపించాయి. "వావ్! ఇది ఎవరి
----------------------------------------
ఒక మేఘం కాలికి ( ) ఆ పొలానికి పరిశీలించాడు. అది ఎల్లప్పుడూ గెలుస్తుంది - పక్షు
----------------------------------------
ఒక చిన్న బో (చిట్టి తల్లిదండ్రులు కనిపెట్టింది! పలకల్ పేరు "లాం
----------------------------------------
ఒక చిన్న ఎలుక బుట్టను కనుగొంది. దాన్ని తెరిచాడు! లోపల బంగారు నాణేలు పెట్టి, దాని కళ్లీ
----------------------------------------
ఒక ముసలి ష్యార్‌లో చిన్న బోలు ఉండేది. దాని పేరు "జ్జు". ఆమె ఎల్లప్పుడూ పేరు "
----------------------------------------
